# Notebook 02 — Preprocessing

## Objective

This notebook performs the preprocessing and quality verification of the standardized datasets used in the UniMTScore framework.

The datasets used in this stage have already undergone initial cleaning and standardization. Therefore, this notebook focuses on preparing the data for the subsequent lexical, syntactic, and semantic evaluation modules.

## Objectives of this Notebook

The main objectives are:

1. Load the standardized datasets.
2. Verify dataset dimensions and column structure.
3. Check for missing and invalid values.
4. Inspect the distribution of languages.
5. Perform safe text normalization.
6. Remove empty or invalid text entries.
7. Verify the range of human evaluation scores.
8. Save the final preprocessed datasets for subsequent evaluation.

## Input Data

The primary evaluation dataset used in this stage is **IndicMT-Eval**, which contains:

- Source sentence
- Reference translation
- Machine-generated translation
- Language
- Normalized MQM score
- Normalized Direct Assessment (DA) score

The project also uses **FLORES-200** and **IN22** as multilingual resources for broader evaluation and analysis.

## Standardized Data Schema

All evaluation records follow a common schema:

| Column | Description |
|--------|-------------|
| `source` | Original source sentence |
| `reference` | Human/reference translation |
| `translation` | Machine-generated translation |
| `language` | Target language |
| `mqm_norm_score` | Normalized human MQM score |
| `da_norm_score` | Normalized human Direct Assessment score |
| `dataset` | Dataset identifier |

## Preprocessing Strategy

The preprocessing applied in this notebook is intentionally conservative.

Only safe text normalization operations are performed, such as:

- Removing leading and trailing whitespace
- Normalizing repeated whitespace
- Removing records with missing essential fields
- Removing exact duplicate records

Aggressive operations such as stop-word removal, stemming, or punctuation removal are avoided because they may alter linguistic information required by the evaluation metrics.

## Output

The final preprocessed datasets are stored for use by the subsequent evaluation modules.

The preprocessing pipeline is:

Standardized Dataset
        ↓
Data Verification
        ↓
Missing Value Check
        ↓
Duplicate Check
        ↓
Safe Text Normalization
        ↓
Score Validation
        ↓
Preprocessed Dataset

These datasets will subsequently be used to calculate the three selected evaluation dimensions of UniMTScore:

- **Lexical:** METEOR
- **Syntactic:** Dependency Similarity
- **Semantic:** COMET

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

INDICMT_DIR = PROJECT_ROOT / "datasets" / "IndicMT_Eval"

print("Files found:")
for p in INDICMT_DIR.iterdir():
    print(p.name)

Files found:
train_clean.csv
validation_clean.csv
test_clean.csv


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

DATA_DIR = PROJECT_ROOT / "processed" / "indicmt"

train = pd.read_csv(DATA_DIR / "train_standardized.csv")
validation = pd.read_csv(DATA_DIR / "validation_standardized.csv")
test = pd.read_csv(DATA_DIR / "test_standardized.csv")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UniMTScore")

print(list((PROJECT_ROOT / "processed").glob("*")))

[PosixPath('/content/drive/MyDrive/UniMTScore/processed/flores'), PosixPath('/content/drive/MyDrive/UniMTScore/processed/indicmt')]


In [ ]:
print(train.shape)
train.head()

(4971, 7)


,source,reference,translation,language,mqm_norm_score,da_norm_score,dataset
0,. Scientists say this animal's plumage was che...,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથી...,એમ.વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીના પીંછા...,Gujarati,0.90,0.8,IndicMT-Eval
1,. Scientists say this animal's plumage was che...,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથી...,. વૈજ્entistsાનિકોકહે છે કે આ પ્રાણીનું પ્લgeમ...,Gujarati,0.60,0.6,IndicMT-Eval
2,. Scientists say this animal's plumage was che...,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથી...,. વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીનું પ્લમે...,Gujarati,0.88,0.8,IndicMT-Eval
3,. Scientists say this animal's plumage was che...,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથી...,વૈજ્ઞાનિકોનું કહેવું છે કે આ પ્રાણીની પલમેજ ઉપ...,Gujarati,0.86,0.8,IndicMT-Eval
4,. Scientists say this animal's plumage was che...,. વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પીંછા ઉપરથી...,વૈજ્ઞાનિકો કહે છે કે આ પ્રાણીનું પ્લુમેજનિસ્તે...,Gujarati,0.76,0.8,IndicMT-Eval


In [ ]:
print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (4971, 7)
Validation: (993, 7)
Test: (2358, 7)


In [ ]:
print("Train columns:")
print(train.columns.tolist())

print("\nMissing values:")
print(train.isnull().sum())

print("\nDuplicate rows:")
print(train.duplicated().sum())

Train columns:
['source', 'reference', 'translation', 'language', 'mqm_norm_score', 'da_norm_score', 'dataset']

Missing values:
source            0
reference         0
translation       0
language          0
mqm_norm_score    0
da_norm_score     0
dataset           0
dtype: int64

Duplicate rows:
0


In [ ]:
from pathlib import Path

INDICMT_DIR = PROJECT_ROOT / "processed" / "indicmt"

print(list(INDICMT_DIR.iterdir()))

[PosixPath('/content/drive/MyDrive/UniMTScore/processed/indicmt/validation_standardized.csv'), PosixPath('/content/drive/MyDrive/UniMTScore/processed/indicmt/test_standardized.csv'), PosixPath('/content/drive/MyDrive/UniMTScore/processed/indicmt/train_standardized.csv')]


In [ ]:
print("Train languages:")
print(train["language"].value_counts())

print("\nValidation languages:")
print(validation["language"].value_counts())

print("\nTest languages:")
print(test["language"].value_counts())

Train languages:
language
Gujarati     998
Malayalam    997
Tamil        995
Marathi      991
Hindi        990
Name: count, dtype: int64

Validation languages:
language
Gujarati     200
Tamil        199
Marathi      198
Malayalam    198
Hindi        198
Name: count, dtype: int64

Test languages:
language
Tamil        276
Hindi        273
Marathi      273
Malayalam    273
Gujarati     272
Maithili     250
Punjabi      247
Assamese     247
Kannada      247
Name: count, dtype: int64


In [ ]:
def clean_text_columns(df):
    df = df.copy()

    text_columns = [
        "source",
        "reference",
        "translation"
    ]

    for col in text_columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    return df

In [ ]:
train = clean_text_columns(train)
validation = clean_text_columns(validation)
test = clean_text_columns(test)

In [ ]:
text_columns = [
    "source",
    "reference",
    "translation"
]

for col in text_columns:
    print(
        col,
        "empty train rows:",
        (train[col].str.len() == 0).sum()
    )


source empty train rows: 0
reference empty train rows: 0
translation empty train rows: 0


In [ ]:
print("MQM range:")
print(train["mqm_norm_score"].min(), train["mqm_norm_score"].max())

print("\nDA range:")
print(train["da_norm_score"].min(), train["da_norm_score"].max())

MQM range:
0.0 1.0

DA range:
0.0 1.0


In [ ]:
from pathlib import Path

PREPROCESS_DIR = PROJECT_ROOT / "processed" / "preprocessed"

PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
train.to_csv(
    PREPROCESS_DIR / "train_preprocessed.csv",
    index=False
)

validation.to_csv(
    PREPROCESS_DIR / "validation_preprocessed.csv",
    index=False
)

test.to_csv(
    PREPROCESS_DIR / "test_preprocessed.csv",
    index=False
)

print("✅ Preprocessed datasets saved.")

✅ Preprocessed datasets saved.


## Conclusion

The preprocessing stage ensures that all evaluation data are consistent, valid, and ready for metric computation.

The resulting datasets provide a reliable input for the next stage of the UniMTScore framework, where lexical, syntactic, and semantic quality features will be extracted and combined into the proposed unified translation evaluation score.